# Mission_Analysis_Test_File
# Generalized pipeline for cubesat post-deployment identification analysis

## Checking the discriminatory strength of BSTAR as a discriminator for the identification and elimination of candidate satellites. 

# Imports 

In [2]:
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timezone
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from sgp4.api import Satrec, SatrecArray

# Constants

In [3]:
ST_BASE  = "https://www.space-track.org"
ST_LOGIN = f"{ST_BASE}/ajaxauth/login"
ST_QUERY = f"{ST_BASE}/basicspacedata/query"

CACHE_DIR = os.path.expanduser("~/mission_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Authentication

In [4]:
def login(username, password, session):
    """Login to Space-Track and return session."""
    resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
    if resp.status_code == 200:
        print(f"Logged in to Space-Track successfully!")
    else:
        print(f"Login failed: {resp.status_code}")
    return session

# TLE Fetching

In [5]:
def fetch_norad_ids(intldes, session, force=False):
    """Fetch NORAD IDs for a launch group from satcat. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_satcat.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"SATCAT loaded from cache — {len(cache['norad_ids'])} objects")
        return cache['norad_ids']
    
    print(f"Querying satcat for {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects   = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'],
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(norad_ids)} payloads for {intldes}")
    return norad_ids


def fetch_current_tles(intldes, norad_ids, session, chunk_size=50, force=False):
    """Fetch current TLEs from gp class. Cached for 1 hour."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_tles.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            print(f"TLEs loaded from cache — {len(cache['tles'])} objects")
            return cache['tles']
        print("TLE cache expired — fetching fresh")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(all_tles)} TLEs for {intldes}")
    return all_tles


def fetch_historical_tles(intldes, norad_ids, launch_date, session,
                           days=60, chunk_size=50, force=False):
    """Fetch historical TLEs from gp_history. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_history.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache — {len(cache['tles'])} records")
        return cache['tles']
    
    # Parse launch date
    launch_dt  = pd.Timestamp(launch_date)
    end_date   = (launch_dt + pd.Timedelta(days=days)).strftime('%Y-%m-%d')
    start_date = launch_dt.strftime('%Y-%m-%d')
    
    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: One-time query — caching permanently")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} "
              f"— {len(resp.json()) if resp.status_code == 200 else 0} records")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {
        'intldes':    intldes,
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Cached {len(all_tles)} historical records")
    return all_tles


In [6]:
def compute_confirmation_timeline(historical_tles, launch_date):
    """
    Compute time to catalog and time to confirmation for each satellite.
    Returns dataframe with cataloging and confirmation timeline.
    """
    hist_df = pd.DataFrame(historical_tles)
    hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
    
    # Confirmed = not TBA or OBJECT
    def is_confirmed(name):
        return not (name.startswith('TBA') or name.startswith('OBJECT'))
    
    hist_df['is_confirmed'] = hist_df['name'].apply(is_confirmed)
    
    launch_ts = pd.Timestamp(launch_date, tz='UTC')
    
    # First cataloged per NORAD
    first_cataloged = (hist_df.groupby('norad')['epoch']
                              .min()
                              .reset_index()
                              .rename(columns={'epoch': 'first_cataloged'}))
    
    # First confirmed per NORAD
    first_confirmed = (hist_df[hist_df['is_confirmed']]
                              .groupby('norad')['epoch']
                              .min()
                              .reset_index()
                              .rename(columns={'epoch': 'first_confirmed'}))
    
    # Unconfirmed name
    tba_names = (hist_df[~hist_df['is_confirmed']]
                        .groupby('norad')['name']
                        .first()
                        .reset_index()
                        .rename(columns={'name': 'unconfirmed_name'}))
    
    # Confirmed name
    confirmed_names = (hist_df[hist_df['is_confirmed']]
                              .groupby('norad')['name']
                              .first()
                              .reset_index()
                              .rename(columns={'name': 'confirmed_name'}))
    
    # Merge all
    df = (first_cataloged
          .merge(first_confirmed,  on='norad', how='left')
          .merge(tba_names,        on='norad', how='left')
          .merge(confirmed_names,  on='norad', how='left'))
    
    df['days_to_catalog']   = (df['first_cataloged'] - launch_ts).dt.total_seconds() / 86400
    df['days_to_confirm']   = (df['first_confirmed'] - launch_ts).dt.total_seconds() / 86400
    df['confirmation_lag']  = df['days_to_confirm'] - df['days_to_catalog']
    df = df.sort_values('days_to_confirm').reset_index(drop=True)
    
    # Summary
    print(f"Confirmation Timeline — {launch_date}")
    print(f"   Total objects:         {len(df)}")
    print(f"   Mean days to catalog:  {df['days_to_catalog'].mean():.1f}")
    print(f"   Mean days to confirm:  {df['days_to_confirm'].mean():.1f}")
    print(f"   Mean confirmation lag: {df['confirmation_lag'].mean():.1f}")
    print(f"   Confirmed:             {df['confirmed_name'].notna().sum()}")
    print(f"   Unconfirmed:           {df['confirmed_name'].isna().sum()}")
    
    return df

# Propagation

In [7]:
def propagate_tles(tles, duration_hrs=24, num_steps=2000):
    """Vectorized SGP4 propagation for all TLEs."""
    now    = Time(datetime.now(timezone.utc))
    times  = now + np.linspace(0, duration_hrs * 3600, num_steps) * u.s
    
    sats   = SatrecArray([Satrec.twoline2rv(t['line1'], t['line2']) for t in tles])
    jd1    = np.array([t.jd1 for t in times])
    jd2    = np.array([t.jd2 for t in times])
    
    e, r_teme, v_teme = sats.sgp4(jd1, jd2)
    
    # Convert to lat/lon/alt
    lats = np.zeros((len(tles), num_steps))
    lons = np.zeros((len(tles), num_steps))
    alts = np.zeros((len(tles), num_steps))
    
    for i in range(len(tles)):
        for j in range(num_steps):
            if e[i, j] != 0:
                continue
            r     = CartesianRepresentation(r_teme[i, j] * u.km)
            teme  = TEME(r, obstime=times[j])
            itrs  = teme.transform_to(ITRS(obstime=times[j]))
            loc   = EarthLocation(*itrs.cartesian.xyz)
            lats[i, j] = loc.lat.deg
            lons[i, j] = loc.lon.deg
            alts[i, j] = loc.height.to(u.km).value
    
    print(f"Propagation complete — {len(tles)} satellites, {num_steps} steps")
    print(f"   Errors: {(e != 0).sum()} non-zero error codes")
    return e, r_teme, v_teme, lats, lons, alts, times

# TLE Dataframe

In [8]:
def parse_bstar(s):
    """Parse compressed scientific notation BSTAR from TLE."""
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            mantissa = float('0.' + s[:i])
            exp      = int(s[i:])
            return mantissa * (10 ** exp)
    return float(s)

def build_tle_dataframe(tles, alts, e):
    """Build analytical dataframe from TLEs and propagated data."""
    if len(tles) == 0:
        print("No TLEs to process")
        return pd.DataFrame()
    
    rows = []
    for i, t in enumerate(tles):
        line1 = t['line1']
        line2 = t['line2']
        
        intldes_raw = line1[9:17].strip()
        year        = intldes_raw[:2]
        num         = intldes_raw[2:5]
        piece       = intldes_raw[5:]
        full_year   = f"19{year}" if int(year) >= 57 else f"20{year}"
        intldes     = f"{full_year}-{num}{piece}"
        
        valid = alts[i][e[i] == 0]
        
        rows.append({
            'idx':          i,
            'name':         t['name'],
            'intldes':      intldes,
            'norad':        line1[2:7].strip(),
            'bstar':        parse_bstar(line1[53:61]),
            'mean_motion':  float(line2[52:63].strip()),
            'inclination':  float(line2[8:16].strip()),
            'eccentricity': float('0.' + line2[26:33].strip()),
            'raan':         float(line2[17:25].strip()),
            'arg_perigee':  float(line2[34:42].strip()),
            'mean_anomaly': float(line2[43:51].strip()),
            'alt_mean':     valid.mean() if len(valid) > 0 else np.nan,
            'alt_min':      valid.min()  if len(valid) > 0 else np.nan,
            'alt_max':      valid.max()  if len(valid) > 0 else np.nan,
            'period_min':   1440 / float(line2[52:63].strip())
        })
    
    df = pd.DataFrame(rows)
    df['alt_range']        = df['alt_max'] - df['alt_min']
    df['decay_rate_km_day'] = [
        np.polyfit(np.linspace(0, 24, alts.shape[1])[e[i] == 0],
                   alts[i][e[i] == 0], 1)[0] * 24
        if (e[i] == 0).sum() > 10 else np.nan
        for i in range(len(tles))
    ]
    return df

# Clustering 

In [40]:
def assign_cluster_dynamic(alt, alt_array):
    """
    Assign cluster based on dynamic percentile boundaries 
    computed from the actual altitude distribution.
    """
    p25 = np.percentile(alt_array, 25)
    p50 = np.percentile(alt_array, 50)
    p75 = np.percentile(alt_array, 75)
    
    if alt < p25:
        return f'A — Lower Quartile (<{p25:.0f}km)'
    elif alt < p50:
        return f'B — Lower Mid ({p25:.0f}-{p50:.0f}km)'
    elif alt < p75:
        return f'C — Upper Mid ({p50:.0f}-{p75:.0f}km)'
    else:
        return f'D — Upper Quartile (>{p75:.0f}km)'


def cluster_dynamic(alt_df):
    """Apply dynamic clustering to a dataframe with alt column."""
    alt_array = alt_df['alt'].values
    alt_df['cluster'] = alt_df['alt'].apply(
        lambda a: assign_cluster_dynamic(a, alt_array)
    )
    
    print(f"   Dynamic cluster boundaries:")
    print(f"   P25: {np.percentile(alt_array, 25):.0f}km")
    print(f"   P50: {np.percentile(alt_array, 50):.0f}km")
    print(f"   P75: {np.percentile(alt_array, 75):.0f}km")
    
    for cluster, group in alt_df.groupby('cluster'):
        print(f"   {cluster} — {len(group)} satellites "
              f"({group['alt'].min():.0f}-{group['alt'].max():.0f}km)")
    
    return alt_df

# BSTAR Correlation

In [73]:
def parse_bstar(s):
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            return float('0.' + s[:i]) * (10 ** int(s[i:]))
    return float(s)

def compute_bstar_correlation(df, exclude_maneuvering=True):
    """Compute BSTAR vs decay rate correlation."""
    df_clean = df.copy()
    
    if exclude_maneuvering:
        df_clean = df_clean[df_clean['alt_range'] < 50]
        print(f"Excluded {len(df) - len(df_clean)} maneuvering satellites")
    
    corr_raw = df_clean['bstar'].corr(df_clean['decay_rate_km_day'])
    r2       = corr_raw ** 2
    
    print(f"BSTAR vs decay rate correlation:")
    print(f"   Pearson r:  {corr_raw:.4f}")
    print(f"   R²:         {r2:.4f}")
    print(f"   Explained variance: {r2*100:.1f}%")
    
    return corr_raw, r2

# RTN Separation

In [11]:
def build_hist_dataframe(historical_tles, launch_date):
    """Build historical TLE dataframe."""
    hist_df = pd.DataFrame(historical_tles)
    hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
    hist_df['date']  = hist_df['epoch'].dt.date
    hist_df['days_since_launch'] = (
        hist_df['epoch'] - pd.Timestamp(launch_date, tz='UTC')
    ).dt.total_seconds() / 86400
    return hist_df


def compute_rtn_separation(hist_df, launch_date, min_sats=5):
    """Compute RTN pairwise separation over time."""
    daily_tles  = (hist_df.sort_values('epoch')
                          .groupby(['norad', 'date'])
                          .first()
                          .reset_index())
    unique_days = sorted(daily_tles['date'].unique())
    
    rtn_results = []
    
    for day in unique_days:
        day_tles = daily_tles[daily_tles['date'] == day]
        day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
        
        if len(day_tles) < min_sats:
            continue
        
        eval_time = Time(f"{day}T12:00:00", scale='utc')
        jd1, jd2  = eval_time.jd1, eval_time.jd2
        
        positions  = []
        velocities = []
        
        for _, row in day_tles.iterrows():
            try:
                sat     = Satrec.twoline2rv(row['line1'], row['line2'])
                e, r, v = sat.sgp4(jd1, jd2)
                if e == 0:
                    positions.append(np.array(r))
                    velocities.append(np.array(v))
            except:
                continue
        
        if len(positions) < min_sats:
            continue
        
        positions  = np.array(positions)
        velocities = np.array(velocities)
        
        r_seps, t_seps, n_seps = [], [], []
        
        for a in range(len(positions)):
            R_hat = positions[a] / np.linalg.norm(positions[a])
            N_hat = np.cross(positions[a], velocities[a])
            N_hat = N_hat / np.linalg.norm(N_hat)
            T_hat = np.cross(N_hat, R_hat)
            T_hat = T_hat / np.linalg.norm(T_hat)
            
            for b in range(a+1, len(positions)):
                dr = positions[b] - positions[a]
                r_seps.append(np.abs(np.dot(dr, R_hat)))
                t_seps.append(np.abs(np.dot(dr, T_hat)))
                n_seps.append(np.abs(np.dot(dr, N_hat)))
        
        days_since = (pd.Timestamp(day) - pd.Timestamp(launch_date)).days
        
        rtn_results.append({
            'date':              day,
            'days_since_launch': days_since,
            'n_sats':            len(positions),
            'mean_R':            np.mean(r_seps),
            'mean_T':            np.mean(t_seps),
            'mean_N':            np.mean(n_seps),
            'median_R':          np.median(r_seps),
            'median_T':          np.median(t_seps),
            'median_N':          np.median(n_seps),
        })
    
    rtn_df = pd.DataFrame(rtn_results)
    rtn_df['rsi_R'] = rtn_df['mean_R'] / rtn_df['mean_R'].iloc[0]
    rtn_df['rsi_T'] = rtn_df['mean_T'] / rtn_df['mean_T'].iloc[0]
    rtn_df['rsi_N'] = rtn_df['mean_N'] / rtn_df['mean_N'].iloc[0]
    
    return rtn_df

# Optimal Window

In [53]:
def compute_optimal_window(rtn_df, min_sats_threshold=None):
    if min_sats_threshold:
        rtn_clean = rtn_df[rtn_df['n_sats'] >= min_sats_threshold].copy()
    else:
        rtn_clean = rtn_df.copy()
    
    if len(rtn_clean) < 3:
        print("Insufficient data for window calculation")
        return None, None, None, rtn_clean
    
    # Smooth mean_R before computing gradient
    rtn_clean['mean_R_smooth'] = rtn_clean['mean_R'].rolling(
        window=5, center=True, min_periods=1
    ).mean()
    
    rtn_clean['radial_growth_rate'] = np.gradient(
        rtn_clean['mean_R_smooth'], rtn_clean['days_since_launch']
    )
    
    # Only look for peak after day 10 to avoid early noise
    rtn_post10 = rtn_clean[rtn_clean['days_since_launch'] >= 10]
    
    if rtn_post10.empty:
        peak_idx = rtn_clean['radial_growth_rate'].idxmax()
    else:
        peak_idx = rtn_post10['radial_growth_rate'].idxmax()
    
    peak_day  = rtn_clean.loc[peak_idx, 'days_since_launch']
    peak_rate = rtn_clean.loc[peak_idx, 'radial_growth_rate']
    half_peak = peak_rate * 0.5
    
    # Only look for 50% decay after peak day
    after_peak = rtn_clean[rtn_clean['days_since_launch'] > peak_day]
    below_half = after_peak[after_peak['radial_growth_rate'] < half_peak]
    
    inflection_day = below_half.iloc[0]['days_since_launch'] if len(below_half) > 0 else None
    
    print(f"Optimal Identification Window:")
    print(f"   Peak growth rate:  {peak_rate:.2f} km/day at day {peak_day}")
    print(f"   50% decay point:   day {inflection_day}")
    print(f"   Optimal window:    days {rtn_clean['days_since_launch'].iloc[0]:.0f} — {inflection_day}")
    
    return peak_day, peak_rate, inflection_day, rtn_clean

# Computing Pairwise Separation

In [128]:
def compute_pairwise_separation(hist_df, cluster_norads, launch_date, min_sats=5):
    """
    Compute daily pairwise separation for all satellite pairs.
    Returns daily fleet stats and per-pair minimum separation.
    """
    daily_tles = (hist_df[hist_df['norad'].isin(cluster_norads)]
                         .sort_values('epoch')
                         .groupby(['norad', 'date'])
                         .last()
                         .reset_index())
    
    unique_days = sorted(daily_tles['date'].unique())
    
    daily_results = []
    pair_min_sep  = {}
    pair_min_day  = {}
    pair_name_map = {}
    
    for day in unique_days:
        day_tles = daily_tles[daily_tles['date'] == day]
        day_tles = day_tles[day_tles['line1'].notna() & 
                            day_tles['line2'].notna()]
        
        if len(day_tles) < min_sats:
            continue
        
        eval_time = Time(f"{day}T12:00:00", scale='utc')
        jd1, jd2  = eval_time.jd1, eval_time.jd2
        
        positions = []
        norads    = []
        names     = []
        
        for _, row in day_tles.iterrows():
            try:
                sat     = Satrec.twoline2rv(row['line1'], row['line2'])
                e, r, v = sat.sgp4(jd1, jd2)
                if e == 0:
                    positions.append(np.array(r))
                    norads.append(row['norad'])
                    names.append(row['name'])
                    pair_name_map[row['norad']] = row['name']
            except:
                continue
        
        if len(positions) < min_sats:
            continue
        
        positions  = np.array(positions)
        days_since = (pd.Timestamp(day) - pd.Timestamp(launch_date)).days
        all_dists  = []
        
        for a in range(len(positions)):
            for b in range(a+1, len(positions)):
                dist     = np.linalg.norm(positions[a] - positions[b])
                pair_key = (norads[a], norads[b])
                all_dists.append(dist)
                
                if pair_key not in pair_min_sep or dist < pair_min_sep[pair_key]:
                    pair_min_sep[pair_key] = dist
                    pair_min_day[pair_key] = days_since
        
        daily_results.append({
            'date':              day,
            'days_since_launch': days_since,
            'n_sats':            len(positions),
            'min_sep':           np.min(all_dists),
            'mean_sep':          np.mean(all_dists),
            'max_sep':           np.max(all_dists),
            'median_sep':        np.median(all_dists),
            'pct10_sep':         np.percentile(all_dists, 10),
        })
    
    daily_df = pd.DataFrame(daily_results)
    
    # Build per-pair dataframe
    pair_rows = []
    for (na, nb), min_sep in pair_min_sep.items():
        pair_rows.append({
            'norad_a':     na,
            'norad_b':     nb,
            'name_a':      pair_name_map.get(na, na),
            'name_b':      pair_name_map.get(nb, nb),
            'min_sep_km':  min_sep,
            'min_sep_day': pair_min_day[(na, nb)]
        })
    
    pair_df = pd.DataFrame(pair_rows).sort_values('min_sep_km')
    
    return daily_df, pair_df


def analyze_pairwise_separation(daily_df, pair_df, sat_quality, 
                                 intldes, optimal_window_end=None):
    """Analyze and plot pairwise separation with quality score context."""
    
    # Plot 1 — daily min/mean/max separation
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=daily_df['days_since_launch'], y=daily_df['min_sep'],
        mode='lines+markers', name='Min Separation',
        line=dict(color='red', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=daily_df['days_since_launch'], y=daily_df['pct10_sep'],
        mode='lines+markers', name='10th Percentile',
        line=dict(color='orange', width=1, dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=daily_df['days_since_launch'], y=daily_df['mean_sep'],
        mode='lines+markers', name='Mean Separation',
        line=dict(color='cyan', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=daily_df['days_since_launch'], y=daily_df['max_sep'],
        mode='lines+markers', name='Max Separation',
        line=dict(color='lime', width=1, dash='dash')
    ))
    
    if optimal_window_end:
        fig.add_vline(x=optimal_window_end, line_dash='dash', 
                      line_color='white',
                      annotation_text=f'Optimal window end (day {optimal_window_end})',
                      annotation_font_color='white')
    
    fig.update_layout(
        title=f'Pairwise Separation Over 60 Days — {intldes}',
        xaxis_title='Days Since Launch',
        yaxis_title='Separation (km)',
        paper_bgcolor='black', plot_bgcolor='black',
        font=dict(color='white'),
        xaxis=dict(gridcolor='gray'),
        yaxis=dict(gridcolor='gray'),
        legend=dict(bgcolor='black')
    )
    #fig.show()
    
    # Plot 2 — histogram of minimum separation per pair
    fig2 = go.Figure()
    fig2.add_trace(go.Histogram(
        x=pair_df['min_sep_km'],
        nbinsx=30,
        marker_color='cyan',
        marker_line=dict(color='black', width=1)
    ))
    fig2.update_layout(
        title=f'Distribution of Minimum Pairwise Separation — {intldes}',
        xaxis_title='Minimum Separation (km)',
        yaxis_title='Number of Pairs',
        paper_bgcolor='black', plot_bgcolor='black',
        font=dict(color='white'),
        xaxis=dict(gridcolor='gray'),
        yaxis=dict(gridcolor='gray')
    )
    #fig2.show()
    
    # Cross-reference with quality scores
    quality_map = dict(zip(sat_quality['norad'], sat_quality['quality_score']))
    flag_map    = dict(zip(sat_quality['norad'], sat_quality['flag']))
    
    pair_df['quality_a'] = pair_df['norad_a'].map(quality_map)
    pair_df['quality_b'] = pair_df['norad_b'].map(quality_map)
    pair_df['flag_a']    = pair_df['norad_a'].map(flag_map)
    pair_df['flag_b']    = pair_df['norad_b'].map(flag_map)
    pair_df['pair_quality'] = (pair_df['quality_a'].fillna(0) +
                                pair_df['quality_b'].fillna(0)) / 2

    flag_display = {
        'good':          'GOOD',
        'low':     'LOW ',
        'outlier': 'OUT '
    }

    print(f"\nMost ambiguous pairs (closest minimum separation):")
    print(f"{'Name A':<25} {'Name B':<25} {'PairQ':>6} {'MinSep':>8} {'Day':>5} {'FlagA':<6} {'FlagB':<6}")
    print("=" * 85)
    for _, row in pair_df.head(15).iterrows():
        flag_a = flag_display.get(row['flag_a'], '?   ')
        flag_b = flag_display.get(row['flag_b'], '?   ')
        print(f"{row['name_a']:<25} {row['name_b']:<25} "
            f"{row['pair_quality']:>6.1f} {row['min_sep_km']:>8.1f} "
            f"{row['min_sep_day']:>5} {flag_a:<6} {flag_b:<6}")
    
    print(f"\nFleet separation summary:")
    print(f"   Min separation ever:    {pair_df['min_sep_km'].min():.1f} km "
          f"at day {pair_df.iloc[0]['min_sep_day']}")
    print(f"   Median min separation:  {pair_df['min_sep_km'].median():.1f} km")
    print(f"   Mean min separation:    {pair_df['min_sep_km'].mean():.1f} km")
    print(f"   Pairs < 100km:          {len(pair_df[pair_df['min_sep_km'] < 100])}")
    print(f"   Pairs < 500km:          {len(pair_df[pair_df['min_sep_km'] < 500])}")
    print(f"   Pairs > 5000km:         {len(pair_df[pair_df['min_sep_km'] > 5000])}")
    
    return pair_df

# Full Pipeline

In [132]:
def run_mission_analysis(intldes, launch_date, session, history_days=60):
    
    print(f"\n{'='*60}")
    print(f"Mission Analysis: {intldes}")
    print(f"Launch Date:      {launch_date}")
    print(f"{'='*60}\n")
    
# Step 1 — fetch NORAD IDs
    print("Step 1 — Fetching NORAD IDs...")
    norad_ids = fetch_norad_ids(intldes, session)
    print(f"   Total payloads cataloged: {len(norad_ids)}")
    
# Step 2 — fetch historical TLEs
    print("\nStep 2 — Fetching historical TLEs...")
    historical_tles = fetch_historical_tles(
        intldes, norad_ids, launch_date, session, days=history_days
    )
    
# Step 3 — build historical dataframe
    print("\nStep 3 — Building historical dataframe...")
    hist_df = build_hist_dataframe(historical_tles, launch_date)
    unique_sats = hist_df['norad'].nunique()
    unique_days = hist_df['date'].nunique()
    print(f"   Total records:       {len(hist_df)}")
    print(f"   Unique satellites:   {unique_sats}")
    print(f"   Days with data:      {unique_days}")
    print(f"   Epoch range:         {hist_df['epoch'].min().date()} → {hist_df['epoch'].max().date()}")
    
# Step 4 — confirmation timeline
    print("\nStep 4 — Computing confirmation timeline...")
    confirm_df = compute_confirmation_timeline(historical_tles, launch_date)
    
# Step 5 — cluster from earliest historical TLEs
    print("\nStep 5 — Clustering from earliest historical TLEs...")
    first_hist = (hist_df.sort_values('epoch')
                         .groupby('norad')
                         .first()
                         .reset_index())
    
    early_alts = []
    for _, row in first_hist.iterrows():
        try:
            sat     = Satrec.twoline2rv(row['line1'], row['line2'])
            t       = Time(row['epoch'])
            e, r, v = sat.sgp4(t.jd1, t.jd2)
            if e == 0:
                alt = np.linalg.norm(r) - 6371.0
                early_alts.append({'norad': row['norad'],
                                    'name':  row['name'],
                                    'alt':   alt})
        except:
            continue
    
    early_alt_df = pd.DataFrame(early_alts)
    alt_array    = early_alt_df['alt'].values
    
    Q1  = np.percentile(alt_array, 25)
    Q3  = np.percentile(alt_array, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    core     = early_alt_df[(early_alt_df['alt'] >= lower_bound) &
                             (early_alt_df['alt'] <= upper_bound)].copy()
    outliers = early_alt_df[(early_alt_df['alt'] < lower_bound) |
                             (early_alt_df['alt'] > upper_bound)].copy()
    
    print(f"   Total satellites:   {len(early_alt_df)}")
    print(f"   Core cluster:       {len(core)} satellites ({core['alt'].min():.0f}-{core['alt'].max():.0f}km)")
    print(f"   Outliers:           {len(outliers)} satellites")
    # Resolve names for outlier satellites
    def resolve_name(norad, hist_df):
        sat_hist  = hist_df[hist_df['norad'] == norad].sort_values('epoch')
        confirmed = sat_hist[~sat_hist['name'].str.startswith('TBA') &
                            ~sat_hist['name'].str.startswith('OBJECT')]
        if len(confirmed) > 0:
            return confirmed.iloc[0]['name']
        return sat_hist.iloc[-1]['name']

    print(f"\n   Outliers (resolved names):")
    for _, row in outliers.iterrows():
        resolved = resolve_name(row['norad'], hist_df)
        print(f"      {row['norad']} — {resolved:<30} {row['alt']:.0f}km")
    
    core['cluster']     = 'Core'
    outliers['cluster'] = 'Outlier'
    early_alt_df        = pd.concat([core, outliers])
    cluster_norads      = core['norad'].tolist()
    hist_cluster        = hist_df[hist_df['norad'].isin(cluster_norads)]
    cluster_focus       = 'Core'
    
    print(f"\n   Focus cluster:      Core")
    print(f"   Satellites:         {len(cluster_norads)}")
    print(f"   Historical records: {len(hist_cluster)}")

# Step 5.5 — BSTAR analysis
    print("\nStep 5.5 — BSTAR analysis...")

    bstar_rows = []
    for norad in cluster_norads:
        sat_hist = hist_df[hist_df['norad'] == norad].sort_values('epoch')
        if len(sat_hist) < 5:
            continue
        
        first = sat_hist.iloc[0]
        last  = sat_hist.iloc[-1]
        
        try:
            first_sat  = Satrec.twoline2rv(first['line1'], first['line2'])
            last_sat   = Satrec.twoline2rv(last['line1'],  last['line2'])
            t_first    = Time(first['epoch'])
            t_last     = Time(last['epoch'])
            
            ef, rf, vf = first_sat.sgp4(t_first.jd1, t_first.jd2)
            el, rl, vl = last_sat.sgp4(t_last.jd1,   t_last.jd2)
            
            if ef != 0 or el != 0:
                continue
            
            alt_first  = np.linalg.norm(rf) - 6371.0
            alt_last   = np.linalg.norm(rl) - 6371.0
            dt_days    = (t_last - t_first).to(u.day).value
            decay_rate = (alt_last - alt_first) / dt_days if dt_days > 0 else np.nan
            
            bstar_rows.append({
                'norad':       norad,
                'name':        first['name'],
                'bstar':       parse_bstar(last['line1'][53:61]),  # most recent BSTAR
                'alt_first':   alt_first,
                'alt_last':    alt_last,
                'total_decay': alt_last - alt_first,
                'decay_rate':  decay_rate,
                'dt_days':     dt_days
            })
        except Exception as ex:
            print(f"   Skipping {norad}: {ex}")
            continue

    if len(bstar_rows) == 0:
        print("   No valid BSTAR data computed")
        bstar_df   = pd.DataFrame()
        corr_daily = np.nan
        corr_total = np.nan
        r2_daily   = np.nan
        r2_total   = np.nan
    else:
        bstar_df    = pd.DataFrame(bstar_rows)
        bstar_clean = bstar_df[bstar_df['total_decay'] < 0].copy()
        
        corr_daily = bstar_clean['bstar'].corr(bstar_clean['decay_rate'])
        r2_daily   = corr_daily ** 2
        corr_total = bstar_clean['bstar'].corr(bstar_clean['total_decay'])
        r2_total   = corr_total ** 2
        
        print(f"   Satellites analyzed:         {len(bstar_df)}")
        print(f"   After removing outliers:     {len(bstar_clean)}")
        print(f"   BSTAR vs day-to-day decay:")
        print(f"      Pearson r:                {corr_daily:.4f}")
        print(f"      R²:                       {r2_daily:.4f}")
        print(f"      Explained variance:       {r2_daily*100:.1f}%")
        print(f"   BSTAR vs total 60-day decay:")
        print(f"      Pearson r:                {corr_total:.4f}")
        print(f"      R²:                       {r2_total:.4f}")
        print(f"      Explained variance:       {r2_total*100:.1f}%")

# Step 5.6 — TLE Propagation Error & Candidate Quality Score
    print("\nStep 5.6 — TLE Propagation Error & Candidate Quality Score...")

    # Build daily TLE sequences
    daily_sequences = {}
    for norad in cluster_norads:
        sat_hist = hist_df[hist_df['norad'] == norad].copy()
        sat_hist['epoch'] = pd.to_datetime(sat_hist['epoch'], utc=True)
        sat_hist['date']  = sat_hist['epoch'].dt.date
        
        daily = (sat_hist.sort_values('epoch')
                        .groupby('date')
                        .last()
                        .reset_index())
        
        if len(daily) < 5:
            continue
        
        daily['days_since_launch'] = [
            (pd.Timestamp(d) - pd.Timestamp(launch_date)).days
            for d in daily['date']
        ]
        
        daily_sequences[norad] = {
            'name':       sat_hist.iloc[-1]['name'],
            'daily_tles': daily[['date', 'days_since_launch',
                                'epoch', 'line1', 'line2']].to_dict('records')
        }

    print(f"   Satellites with daily sequences: {len(daily_sequences)}")

    def get_rtn_error(r_prop, r_actual, v_ref):
        """Decompose position error into RTN components."""
        dr    = np.array(r_prop) - np.array(r_actual)
        r_vec = np.array(r_actual)
        v_vec = np.array(v_ref)
        R_hat = r_vec / np.linalg.norm(r_vec)
        N_hat = np.cross(r_vec, v_vec)
        N_hat = N_hat / np.linalg.norm(N_hat)
        T_hat = np.cross(N_hat, R_hat)
        return {
            'total':       np.linalg.norm(dr),
            'radial':      abs(np.dot(dr, R_hat)),
            'along_track': abs(np.dot(dr, T_hat)),
            'cross_track': abs(np.dot(dr, N_hat))
        }

    k_days       = [1, 5, 7, 14]
    daily_errors = []

    for norad, data in daily_sequences.items():
        tles   = data['daily_tles']
        n_days = len(tles)
        
        for n in range(min(14, n_days - 1)):
            try:
                xn      = tles[n]
                sat_n   = Satrec.twoline2rv(xn['line1'], xn['line2'])
                day_n   = xn['days_since_launch']
                
                xn1     = tles[n + 1]
                sat_n1  = Satrec.twoline2rv(xn1['line1'], xn1['line2'])
                t_n1    = Time(xn1['epoch'])
                dt_days = xn1['days_since_launch'] - day_n
                
                if dt_days <= 0:
                    continue
                
                en1, rn1, vn1 = sat_n1.sgp4(t_n1.jd1, t_n1.jd2)
                e0,  r0,  v0  = sat_n.sgp4(t_n1.jd1,  t_n1.jd2)
                
                if en1 != 0 or e0 != 0:
                    continue
                
                local_rtn = get_rtn_error(r0, rn1, vn1)
                
                row = {
                    'norad':            norad,
                    'name':             data['name'],
                    'n':                n,
                    'day_n':            day_n,
                    'dt_local':         dt_days,
                    'local_error_norm': local_rtn['total']       / dt_days,
                    'local_T_norm':     local_rtn['along_track'] / dt_days,
                    'local_R_norm':     local_rtn['radial']      / dt_days,
                    'local_N_norm':     local_rtn['cross_track'] / dt_days,
                }
                
                for k in k_days:
                    target_day = day_n + k
                    candidates = [t for t in tles
                                if abs(t['days_since_launch'] - target_day) <= 1]
                    if not candidates:
                        for comp in ['total', 'T', 'R', 'N']:
                            row[f'global_{comp}_day{k}'] = np.nan
                        continue
                    
                    xk    = min(candidates,
                                key=lambda x: abs(x['days_since_launch'] - target_day))
                    sat_k = Satrec.twoline2rv(xk['line1'], xk['line2'])
                    t_k   = Time(xk['epoch'])
                    
                    ek,  rk,  vk  = sat_k.sgp4(t_k.jd1, t_k.jd2)
                    e0k, r0k, v0k = sat_n.sgp4(t_k.jd1, t_k.jd2)
                    
                    if ek != 0 or e0k != 0:
                        for comp in ['total', 'T', 'R', 'N']:
                            row[f'global_{comp}_day{k}'] = np.nan
                        continue
                    
                    g_rtn = get_rtn_error(r0k, rk, vk)
                    row[f'global_total_day{k}'] = g_rtn['total']
                    row[f'global_T_day{k}']     = g_rtn['along_track']
                    row[f'global_R_day{k}']     = g_rtn['radial']
                    row[f'global_N_day{k}']     = g_rtn['cross_track']
                
                daily_errors.append(row)
            except:
                continue

    daily_err_df = pd.DataFrame(daily_errors)
    print(f"   Total observations: {len(daily_err_df)}")

    # Correlation analysis
    print(f"\n   Predictive Power (Local Norm Error vs Global Error):")
    print(f"   {'Metric':<25} {'Day 1':>8} {'Day 5':>8} {'Day 7':>8} {'Day 14':>8}")
    print(f"   {'-'*60}")

    for component, label in [('total', 'Total Error'),
                            ('T',     'Along-Track'),
                            ('R',     'Radial'),
                            ('N',     'Cross-Track')]:
        r_vals = []
        for k in k_days:
            col   = f'global_{component}_day{k}'
            clean = daily_err_df[['local_error_norm', col]].dropna()
            clean = clean[(clean['local_error_norm'] > 0) & (clean[col] > 0)]
            if len(clean) < 10:
                r_vals.append(np.nan)
                continue
            r = clean['local_error_norm'].corr(clean[col])
            r_vals.append(r)
        print(f"   {label:<25} {r_vals[0]:>8.3f} {r_vals[1]:>8.3f} "
            f"{r_vals[2]:>8.3f} {r_vals[3]:>8.3f}")

    # Candidate quality score
    sat_quality = (daily_err_df.groupby(['norad', 'name'])
                                ['local_error_norm']
                                .agg(['mean', 'median', 'std'])
                                .reset_index())
    sat_quality.columns = ['norad', 'name', 'mean_local_err',
                            'median_local_err', 'std_local_err']

        # Outlier threshold — 3x fleet median
    fleet_median   = sat_quality['mean_local_err'].median()
    outlier_thresh = 3 * fleet_median

    # Split
    outliers_q = sat_quality[sat_quality['mean_local_err'] > outlier_thresh]
    core_q     = sat_quality[sat_quality['mean_local_err'] <= outlier_thresh]

    # P95 from core only
    p95 = core_q['mean_local_err'].quantile(0.95)

    # Score — outliers get 0, core gets 0-100
    sat_quality['quality_score'] = sat_quality.apply(
        lambda row: 0.0 if row['mean_local_err'] > outlier_thresh
        else round(100 * (1 - min(row['mean_local_err'], p95) / p95), 1),
        axis=1
    )

    # Flag — outlier OR bottom 10% of core
    bottom_thresh = core_q['mean_local_err'].quantile(0.90)
    sat_quality['flag'] = sat_quality.apply(
        lambda row: 'outlier'  if row['mean_local_err'] > outlier_thresh
        else        'low'      if row['mean_local_err'] > bottom_thresh
        else        'good',
        axis=1
    )

    sat_quality = sat_quality.sort_values('quality_score', ascending=False)

    print(f"\n   Candidate Quality Scores:")
    print(f"   {'NORAD':<8} {'Name':<30} {'Mean Err':>10} {'Quality':>10} {'Flag'}")
    print(f"   {'-'*70}")
    for _, row in sat_quality.iterrows():
        print(f"   {row['norad']:<8} {row['name']:<30} "
            f"{row['mean_local_err']:>10.3f} {row['quality_score']:>10.1f} {row['flag']}")

    print(f"\n   Fleet quality summary:")
    print(f"   Fleet median error:     {fleet_median:.3f} km/day")
    print(f"   Outlier threshold:      {outlier_thresh:.3f} km/day (3x median)")
    print(f"   P95 normalization cap:  {p95:.3f} km/day")
    print(f"   Outliers flagged:       {len(outliers_q)}")
    print(f"   Low quality flagged:    {len(sat_quality[sat_quality['flag'] == 'low'])}")
    print(f"   Core candidates:        {len(core_q)}")    

# Step 5.7 — Pairwise minimum separation
    print("\nStep 5.7 — Pairwise minimum separation...")

    # Filter hist_cluster to first 30 days — identification window
    hist_window = hist_cluster[hist_cluster['days_since_launch'] <= 30]

    daily_sep_df, pair_df = compute_pairwise_separation(
        hist_window, cluster_norads, launch_date
    )
    pair_df = analyze_pairwise_separation(
        daily_sep_df, pair_df, sat_quality, intldes,
        optimal_window_end=None  # updated after optimal window computed
    )
# Step 6 — RTN separation
    print("\nStep 6 — Computing RTN separation...")
    rtn_df = compute_rtn_separation(hist_cluster, launch_date)
    
    if rtn_df.empty:
        print("Insufficient data for RTN analysis")
        return None
    
# Step 7 — optimal window
    print("\nStep 7 — Computing optimal identification window...")
    min_thresh = max(5, int(len(cluster_norads) * 0.85))
    peak_day, peak_rate, inflection_day, rtn_clean = compute_optimal_window(
        rtn_df, min_sats_threshold=min_thresh
    )
    
# Step 8 — plot
    print("\nStep 8 — Plotting RTN separation...")
    plot_rtn(rtn_df, intldes)
    
    print(f"\n{'='*60}")
    print(f"Summary: {intldes}")
    print(f"   Total payloads:        {len(norad_ids)}")
    print(f"   Unique sats in hist:   {unique_sats}")
    print(f"   Days with TLE data:    {unique_days}")
    print(f"   Mean days to catalog:  {confirm_df['days_to_catalog'].mean():.1f}")
    print(f"   Mean days to confirm:  {confirm_df['days_to_confirm'].mean():.1f}")
    print(f"   Confirmation lag:      {confirm_df['confirmation_lag'].mean():.1f}")
    print(f"   Focus cluster:         {cluster_focus}")
    print(f"   Cluster satellites:    {len(cluster_norads)}")
    print(f"   Optimal window:        days {rtn_df['days_since_launch'].iloc[0]:.0f} — {inflection_day}")
    print(f"   BSTAR daily corr:      {corr_daily:.4f} (R²={r2_daily:.3f})")
    print(f"   BSTAR total corr:      {corr_total:.4f} (R²={r2_total:.3f})")
    print(f"{'='*60}\n")
    
    return {
        'intldes':          intldes,
        'launch_date':      launch_date,
        'norad_ids':        norad_ids,
        'hist_df':          hist_df,
        'confirm_df':       confirm_df,
        'early_alt_df':     early_alt_df,
        'rtn_df':           rtn_df,
        'rtn_clean':        rtn_clean,
        'cluster_focus':    cluster_focus,
        'cluster_norads':   cluster_norads,
        'peak_day':         peak_day,
        'peak_rate':        peak_rate,
        'inflection_day':   inflection_day,
        'bstar_df':         bstar_df,
        'corr_daily':       corr_daily,
        'corr_total':       corr_total,
        'r2_daily':         r2_daily,
        'r2_total':    r2_total,
        'daily_err_df': daily_err_df,
        'sat_quality':  sat_quality,
        'outliers_q':   outliers_q,
        'daily_sep_df': daily_sep_df,
        'pair_df':      pair_df,
    }

# Running Mission Analysis 

In [133]:
import requests
session = requests.Session()
session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

results_t7 = run_mission_analysis('2023-054', '2023-04-15', session,
                          history_days=60) 


Mission Analysis: 2023-054
Launch Date:      2023-04-15

Step 1 — Fetching NORAD IDs...
SATCAT loaded from cache — 48 objects
   Total payloads cataloged: 48

Step 2 — Fetching historical TLEs...
Historical TLEs loaded from cache — 6565 records

Step 3 — Building historical dataframe...
   Total records:       6565
   Unique satellites:   47
   Days with data:      58
   Epoch range:         2023-04-15 → 2023-06-13

Step 4 — Computing confirmation timeline...
Confirmation Timeline — 2023-04-15
   Total objects:         47
   Mean days to catalog:  8.9
   Mean days to confirm:  21.2
   Mean confirmation lag: 11.6
   Confirmed:             41
   Unconfirmed:           6

Step 5 — Clustering from earliest historical TLEs...
   Total satellites:   47
   Core cluster:       36 satellites (515-516km)
   Outliers:           11 satellites

   Outliers (resolved names):
      56178 — IMECE                          689km
      56215 — VCUB1                          695km
      56216 — ELO-3    


Summary: 2023-054
   Total payloads:        48
   Unique sats in hist:   47
   Days with TLE data:    58
   Mean days to catalog:  8.9
   Mean days to confirm:  21.2
   Confirmation lag:      11.6
   Focus cluster:         Core
   Cluster satellites:    36
   Optimal window:        days 3 — 22
   BSTAR daily corr:      -0.5820 (R²=0.339)
   BSTAR total corr:      -0.5805 (R²=0.337)



# Mission Summaries

In [134]:
def build_mission_summary(results):
    """Extract key metrics from run_mission_analysis results into a summary dict."""
    
    confirm_df  = results['confirm_df']
    bstar_df    = results['bstar_df']
    sat_quality = results['sat_quality']
    rtn_df      = results['rtn_df']
    pair_df     = results['pair_df']
    early_alt_df = results['early_alt_df']
    daily_err_df = results['daily_err_df']
    
    # Propagation predictive power at key days
    def get_corr(daily_err_df, k):
        col   = f'global_total_day{k}'
        clean = daily_err_df[['local_error_norm', col]].dropna()
        clean = clean[(clean['local_error_norm'] > 0) & (clean[col] > 0)]
        if len(clean) < 10:
            return np.nan
        return clean['local_error_norm'].corr(clean[col])
    
    # Quality counts
    good_count    = len(sat_quality[sat_quality['flag'] == 'good'])
    low_count     = len(sat_quality[sat_quality['flag'] == 'low'])
    outlier_count = len(sat_quality[sat_quality['flag'] == 'outlier'])
    
    # Core altitude range
    core = early_alt_df[early_alt_df['cluster'] == 'Core']
    
    summary = {
        # Mission metadata
        'Designator':          results['intldes'],
        'Launch Date':         results['launch_date'],
        'Total Payloads':      len(results['norad_ids']),
        'Hist Satellites':     results['hist_df']['norad'].nunique(),
        'Days TLE Data':       results['hist_df']['date'].nunique(),
        
        # Confirmation timeline
        'Days to Catalog':     round(confirm_df['days_to_catalog'].mean(), 1),
        'Days to Confirm':     round(confirm_df['days_to_confirm'].mean(), 1),
        'Confirm Lag':         round(confirm_df['confirmation_lag'].mean(), 1),
        
        # Clustering
        'Core Cluster N':      len(core),
        'Alt Range (km)':      f"{core['alt'].min():.0f}-{core['alt'].max():.0f}",
        'Outliers':            len(early_alt_df[early_alt_df['cluster'] == 'Outlier']),
        
        # BSTAR
        'BSTAR daily r':       round(results['corr_daily'], 3),
        'BSTAR total r':       round(results['corr_total'], 3),
        'BSTAR R²':            round(results['r2_total'], 3),
        
        # Propagation predictability
        'Prop r day5':         round(get_corr(daily_err_df, 5), 3),
        'Prop r day7':         round(get_corr(daily_err_df, 7), 3),
        'Prop r day14':        round(get_corr(daily_err_df, 14), 3),
        
        # Quality scores
        'Good Candidates':     good_count,
        'Low Quality':         low_count,
        'Outlier Candidates':  outlier_count,
        
        # RTN
        'Peak Growth (km/day)': round(results['peak_rate'], 1),
        'Optimal Window':       f"days {rtn_df['days_since_launch'].iloc[0]:.0f}-{results['inflection_day']}",
        
        # Pairwise separation
        'Min Sep (km)':        round(pair_df['min_sep_km'].min(), 1),
        'Pairs < 100km':       len(pair_df[pair_df['min_sep_km'] < 100]),
    }
    
    return summary


def print_mission_summary(summary):
    """Print a formatted mission summary."""
    print(f"\n{'='*60}")
    print(f"MISSION SUMMARY — {summary['Designator']}")
    print(f"{'='*60}")
    
    print(f"\n--- Mission Metadata ---")
    print(f"   Launch Date:           {summary['Launch Date']}")
    print(f"   Total Payloads:        {summary['Total Payloads']}")
    print(f"   Historical Satellites: {summary['Hist Satellites']}")
    print(f"   Days TLE Data:         {summary['Days TLE Data']}")
    
    print(f"\n--- Confirmation Timeline ---")
    print(f"   Mean Days to Catalog:  {summary['Days to Catalog']}")
    print(f"   Mean Days to Confirm:  {summary['Days to Confirm']}")
    print(f"   Confirmation Lag:      {summary['Confirm Lag']} days")
    
    print(f"\n--- Clustering ---")
    print(f"   Core Cluster:          {summary['Core Cluster N']} satellites")
    print(f"   Altitude Range:        {summary['Alt Range (km)']} km")
    print(f"   Outliers:              {summary['Outliers']}")
    
    print(f"\n--- BSTAR Discrimination ---")
    print(f"   Daily r:               {summary['BSTAR daily r']}")
    print(f"   Total decay r:         {summary['BSTAR total r']}")
    print(f"   R²:                    {summary['BSTAR R²']}")
    
    print(f"\n--- Propagation Predictability ---")
    print(f"   Local -> Day 5:        r={summary['Prop r day5']}")
    print(f"   Local -> Day 7:        r={summary['Prop r day7']}")
    print(f"   Local -> Day 14:       r={summary['Prop r day14']}")
    
    print(f"\n--- Candidate Quality ---")
    print(f"   Good:                  {summary['Good Candidates']}")
    print(f"   Low quality:           {summary['Low Quality']}")
    print(f"   Outliers:              {summary['Outlier Candidates']}")
    
    print(f"\n--- RTN Separation ---")
    print(f"   Peak Growth Rate:      {summary['Peak Growth (km/day)']} km/day")
    print(f"   Optimal Window:        {summary['Optimal Window']}")
    
    print(f"\n--- Pairwise Separation ---")
    print(f"   Min Separation:        {summary['Min Sep (km)']} km")
    print(f"   Pairs < 100km:         {summary['Pairs < 100km']}")
    print(f"{'='*60}\n")

In [135]:
summary_t7 = build_mission_summary(results_t7)
print_mission_summary(summary_t7)


MISSION SUMMARY — 2023-054

--- Mission Metadata ---
   Launch Date:           2023-04-15
   Total Payloads:        48
   Historical Satellites: 47
   Days TLE Data:         58

--- Confirmation Timeline ---
   Mean Days to Catalog:  8.9
   Mean Days to Confirm:  21.2
   Confirmation Lag:      11.6 days

--- Clustering ---
   Core Cluster:          36 satellites
   Altitude Range:        515-516 km
   Outliers:              11

--- BSTAR Discrimination ---
   Daily r:               -0.582
   Total decay r:         -0.581
   R²:                    0.337

--- Propagation Predictability ---
   Local -> Day 5:        r=0.797
   Local -> Day 7:        r=0.784
   Local -> Day 14:       r=0.615

--- Candidate Quality ---
   Good:                  28
   Low quality:           4
   Outliers:              4

--- RTN Separation ---
   Peak Growth Rate:      370.0 km/day
   Optimal Window:        days 3-22

--- Pairwise Separation ---
   Min Separation:        0.6 km
   Pairs < 100km:         62


In [137]:
def build_comparison_table(summaries):
    """Build cross-mission comparison dataframe."""
    df = pd.DataFrame(summaries)
    print(df.to_string(index=False))
    return df

comparison_df = build_comparison_table([summary_t7])

Designator Launch Date  Total Payloads  Hist Satellites  Days TLE Data  Days to Catalog  Days to Confirm  Confirm Lag  Core Cluster N Alt Range (km)  Outliers  BSTAR daily r  BSTAR total r  BSTAR R²  Prop r day5  Prop r day7  Prop r day14  Good Candidates  Low Quality  Outlier Candidates  Peak Growth (km/day) Optimal Window  Min Sep (km)  Pairs < 100km
  2023-054  2023-04-15              48               47             58              8.9             21.2         11.6              36        515-516        11         -0.582         -0.581     0.337        0.797        0.784         0.615               28            4                   4                 370.0      days 3-22           0.6             62


# Running "Finalized Mission Analysis" 

In [140]:
#results_t6 = run_mission_analysis('2023-001', '2023-01-03', session, history_days=60)
#results_t7 = run_mission_analysis('2023-001', '2023-01-03', session, history_days=60)
results_t8 = run_mission_analysis('2023-084', '2023-06-12', session, history_days=60)
#results_t9 = run_mission_analysis('2023-001', '2023-01-03', session, history_days=60)




Mission Analysis: 2023-084
Launch Date:      2023-06-12

Step 1 — Fetching NORAD IDs...
Querying satcat for 2023-084...
Fetched 66 payloads for 2023-084
   Total payloads cataloged: 66

Step 2 — Fetching historical TLEs...
Querying gp_history: 2023-06-12 → 2023-08-11
   Chunk 1: status 200 — 2680 records
   Chunk 2: status 200 — 741 records
Cached 3421 historical records

Step 3 — Building historical dataframe...
   Total records:       3421
   Unique satellites:   64
   Days with data:      51
   Epoch range:         2023-06-20 → 2023-08-10

Step 4 — Computing confirmation timeline...
Confirmation Timeline — 2023-06-12
   Total objects:         64
   Mean days to catalog:  11.5
   Mean days to confirm:  21.8
   Mean confirmation lag: 10.3
   Confirmed:             57
   Unconfirmed:           7

Step 5 — Clustering from earliest historical TLEs...
   Total satellites:   64
   Core cluster:       55 satellites (545-546km)
   Outliers:           9 satellites

   Outliers (resolved name


Summary: 2023-084
   Total payloads:        66
   Unique sats in hist:   64
   Days with TLE data:    51
   Mean days to catalog:  11.5
   Mean days to confirm:  21.8
   Confirmation lag:      10.3
   Focus cluster:         Core
   Cluster satellites:    55
   Optimal window:        days 8 — 22
   BSTAR daily corr:      -0.1861 (R²=0.035)
   BSTAR total corr:      -0.1653 (R²=0.027)



In [143]:
results_t9 = run_mission_analysis('2023-174', '2023-11-11', session, history_days=60)


Mission Analysis: 2023-174
Launch Date:      2023-11-11

Step 1 — Fetching NORAD IDs...
Querying satcat for 2023-174...
Fetched 95 payloads for 2023-174
   Total payloads cataloged: 95

Step 2 — Fetching historical TLEs...
Querying gp_history: 2023-11-11 → 2024-01-10
   Chunk 1: status 200 — 5645 records
   Chunk 2: status 200 — 4328 records
Cached 9973 historical records

Step 3 — Building historical dataframe...
   Total records:       9973
   Unique satellites:   92
   Days with data:      43
   Epoch range:         2023-11-28 → 2024-01-09

Step 4 — Computing confirmation timeline...
Confirmation Timeline — 2023-11-11
   Total objects:         92
   Mean days to catalog:  19.9
   Mean days to confirm:  29.2
   Mean confirmation lag: 11.0
   Confirmed:             76
   Unconfirmed:           16

Step 5 — Clustering from earliest historical TLEs...
   Total satellites:   92
   Core cluster:       81 satellites (536-542km)
   Outliers:           11 satellites

   Outliers (resolved n


Summary: 2023-174
   Total payloads:        95
   Unique sats in hist:   92
   Days with TLE data:    43
   Mean days to catalog:  19.9
   Mean days to confirm:  29.2
   Confirmation lag:      11.0
   Focus cluster:         Core
   Cluster satellites:    81
   Optimal window:        days 17 — 31
   BSTAR daily corr:      -0.5947 (R²=0.354)
   BSTAR total corr:      -0.4448 (R²=0.198)

